# 测试报错信息（GTA）
- 调查什么图报错了，所以batch size为1

In [1]:
import argparse
from tqdm import tqdm
import dgl
import torch
import torch.nn.functional as F
from dgl.data import tu
from model.encoder import DiffPool

def prepare_data(dataset, shuffle=False, prog_args=None):
    """
    preprocess TU dataset according to DiffPool's paper setting and load dataset into dataloader
    """
    return dgl.dataloading.GraphDataLoader(
        dataset,
        batch_size=prog_args.batch_size,
        shuffle=shuffle,
        num_workers=prog_args.n_worker,
    )


def train(dataset, model, prog_args, id_card_protein):
    """
    training function
    """
    dataloader = dataset

    if prog_args.cuda > 0:
        torch.cuda.set_device(0)
    
    for epoch in range(prog_args.epoch):
        model.train()
        print("\nEPOCH ###### {} ######".format(epoch))
        for batch_idx, (batch_graph, graph_labels) in tqdm(enumerate(dataloader), total=len(dataloader)):
            for key, value in batch_graph.ndata.items():
                batch_graph.ndata[key] = value.float()
            graph_labels = graph_labels.long()
            if torch.cuda.is_available():
                batch_graph = batch_graph.to(torch.cuda.current_device())
                graph_labels = graph_labels.cuda()

            model.zero_grad()
            # ypred = model(batch_graph)
            temp = batch_graph
            protein_temp_id_card = ''
            protein_temp_id_card = protein_temp_id_card + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][0][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][1][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][2][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][6]), 5))
            try:
                ypred = model(batch_graph)
            except:
                print('This is the protein name not be forward ==========', id_card_protein[protein_temp_id_card])
                continue
            try:
                ttt = id_card_protein[protein_temp_id_card]
            except:
                print('This is the data not be traced ==========', protein_temp_id_card)
        torch.cuda.empty_cache()
        break
    return 'Trian successfully'


/home/admin123/software/anaconda3/envs/dgl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
sample_redies_udp = 6
sample_redies_sugar = 6


global_train_time_per_epoch = []

print("{:=^100}".format('prog_args'))
global_train_time_per_epoch = []
prog_args = argparse.Namespace(dataset=f'GTmining_6_6', pool_ratio=0.10, num_pool=1, cuda=1, lr=1e-4, clip=1.0,
                               batch_size=1, epoch=500, train_ratio=0.7, test_ratio=0.1, n_worker=2, gc_per_block=3,
                               dropout=0.20, method="diffpool", bn=True, bias=True, save_dir="./model_param",
                               load_epoch=-1, data_mode="default", linkpred=False, hidden_dim=64, embedding_dim=64)
print(prog_args)

print("{:=^100}".format('加载数据'))
dataset_train = tu.RhaFinderDataset(name="GTmining",
                                    raw_dir=f'../data/dl_data/GTA/fold1/train/')
dataset_validation = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/GTA/fold1/validation/')
dataset_test = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/GTA/fold1/test/')
dataset_nova = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/GTA/fold1/nova/')
train_dataloader = prepare_data(dataset_train, shuffle=True, prog_args=prog_args)
validation_dataloader = prepare_data(dataset_validation, shuffle=False, prog_args=prog_args)
test_dataloader = prepare_data(dataset_test, shuffle=False, prog_args=prog_args)
nova_dataloader = prepare_data(dataset_nova, shuffle=False, prog_args=prog_args)

input_dim_train, label_dim_train, max_num_node_train = dataset_train.statistics()
input_dim_validation, label_dim_validation, max_num_node_validation = dataset_validation.statistics()
input_dim_test, label_dim_test, max_num_node_test = dataset_test.statistics()
input_dim_nova, label_dim_nova, max_num_node_nova = dataset_nova.statistics()
max_num_node = max([max_num_node_train, max_num_node_validation, max_num_node_test, max_num_node_nova])
input_dim = input_dim_train
label_dim = label_dim_train
print("++++++++++ STATISTICS ABOUT THE DATASET ++++++++++")
print("dataset feature dimension is", input_dim_train)
print("dataset label dimension is", label_dim_train)
print("the max num node is", max_num_node)
print("number of graphs is", len(dataset_train) + len(dataset_validation)+ len(dataset_test)+ len(dataset_nova))

hidden_dim = prog_args.hidden_dim  # used to be 64
embedding_dim = prog_args.embedding_dim

assign_dim = int(max_num_node * prog_args.pool_ratio)
print("++++++++++MODEL STATISTICS++++++++")
print("model hidden dim is", hidden_dim)
print("model embedding dim for graph instance embedding", embedding_dim)
print("initial batched pool graph dim is", assign_dim)
activation = F.relu

# initialize model
model = DiffPool(
    input_dim,
    hidden_dim,
    embedding_dim,
    label_dim,
    activation,
    prog_args.gc_per_block,
    prog_args.dropout,
    prog_args.num_pool,
    prog_args.linkpred,
    prog_args.batch_size,
    "meanpool",
    assign_dim,
    [],
    prog_args.pool_ratio,
)

print("model init finished")
print("MODEL:::::::", prog_args.method)
if prog_args.cuda:
    model = model.cuda()

In [28]:
# train_dataloader  validation_dataloader  test_dataloader  nova_dataloader
# train  validation  test  nova

id_card_protein = {}
with open(f'../data/dl_data/GTA/fold1/nova/Predict_correspond_information.txt', 'r')as f:
    for dd in f.readlines():
        dd = dd.split('\n')[0].split('===')
        id_card_protein[dd[1]] = dd[0]

logger = train(
    nova_dataloader, model, prog_args, id_card_protein
)

print("Ending!!!")


EPOCH ###### 0 ######


100%|██████████| 475/475 [00:16<00:00, 28.96it/s]

Ending!!!


# 测试报错信息（GTB）
- 调查什么图报错了，所以batch size为1

In [1]:
import argparse
from tqdm import tqdm
import dgl
import torch
import torch.nn.functional as F
from dgl.data import tu
from model.encoder import DiffPool

def prepare_data(dataset, shuffle=False, prog_args=None):
    """
    preprocess TU dataset according to DiffPool's paper setting and load dataset into dataloader
    """
    return dgl.dataloading.GraphDataLoader(
        dataset,
        batch_size=prog_args.batch_size,
        shuffle=shuffle,
        num_workers=prog_args.n_worker,
    )


def train(dataset, model, prog_args, id_card_protein):
    """
    training function
    """
    dataloader = dataset

    if prog_args.cuda > 0:
        torch.cuda.set_device(0)
    
    for epoch in range(prog_args.epoch):
        model.train()
        print("\nEPOCH ###### {} ######".format(epoch))
        for batch_idx, (batch_graph, graph_labels) in tqdm(enumerate(dataloader), total=len(dataloader)):
            for key, value in batch_graph.ndata.items():
                batch_graph.ndata[key] = value.float()
            graph_labels = graph_labels.long()
            if torch.cuda.is_available():
                batch_graph = batch_graph.to(torch.cuda.current_device())
                graph_labels = graph_labels.cuda()

            model.zero_grad()
            # ypred = model(batch_graph)
            temp = batch_graph
            protein_temp_id_card = ''
            protein_temp_id_card = protein_temp_id_card + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][0][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][1][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][2][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][6]), 5))
            try:
                ypred = model(batch_graph)
            except:
                print('This is the protein name not be forward ==========', id_card_protein[protein_temp_id_card])
                continue
            try:
                ttt = id_card_protein[protein_temp_id_card]
            except:
                print('This is the data not be traced ==========', protein_temp_id_card)
        torch.cuda.empty_cache()
        break
    return 'Trian successfully'


/home/admin123/software/anaconda3/envs/dgl/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
sample_redies_udp = 6
sample_redies_sugar = 6


global_train_time_per_epoch = []

print("{:=^100}".format('prog_args'))
global_train_time_per_epoch = []
prog_args = argparse.Namespace(dataset=f'GTmining_6_6', pool_ratio=0.10, num_pool=1, cuda=1, lr=1e-4, clip=1.0,
                               batch_size=1, epoch=500, train_ratio=0.7, test_ratio=0.1, n_worker=2, gc_per_block=3,
                               dropout=0.20, method="diffpool", bn=True, bias=True, save_dir="./model_param",
                               load_epoch=-1, data_mode="default", linkpred=False, hidden_dim=64, embedding_dim=64)
print(prog_args)

print("{:=^100}".format('加载数据'))
dataset_train = tu.RhaFinderDataset(name="GTmining",
                                    raw_dir=f'../data/dl_data/GTB/fold1/train/')
dataset_validation = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/GTB/fold1/validation/')
dataset_test = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/GTB/fold1/test/')
dataset_nova = tu.RhaFinderDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/GTB/fold1/nova/')
train_dataloader = prepare_data(dataset_train, shuffle=True, prog_args=prog_args)
validation_dataloader = prepare_data(dataset_validation, shuffle=False, prog_args=prog_args)
test_dataloader = prepare_data(dataset_test, shuffle=False, prog_args=prog_args)
nova_dataloader = prepare_data(dataset_nova, shuffle=False, prog_args=prog_args)

input_dim_train, label_dim_train, max_num_node_train = dataset_train.statistics()
input_dim_validation, label_dim_validation, max_num_node_validation = dataset_validation.statistics()
input_dim_test, label_dim_test, max_num_node_test = dataset_test.statistics()
input_dim_nova, label_dim_nova, max_num_node_nova = dataset_nova.statistics()
max_num_node = max([max_num_node_train, max_num_node_validation, max_num_node_test, max_num_node_nova])
input_dim = input_dim_train
label_dim = label_dim_train
print("++++++++++ STATISTICS ABOUT THE DATASET ++++++++++")
print("dataset feature dimension is", input_dim_train)
print("dataset label dimension is", label_dim_train)
print("the max num node is", max_num_node)
print("number of graphs is", len(dataset_train) + len(dataset_validation)+ len(dataset_test)+ len(dataset_nova))

hidden_dim = prog_args.hidden_dim  # used to be 64
embedding_dim = prog_args.embedding_dim

assign_dim = int(max_num_node * prog_args.pool_ratio)
print("++++++++++MODEL STATISTICS++++++++")
print("model hidden dim is", hidden_dim)
print("model embedding dim for graph instance embedding", embedding_dim)
print("initial batched pool graph dim is", assign_dim)
activation = F.relu

# initialize model
model = DiffPool(
    input_dim,
    hidden_dim,
    embedding_dim,
    label_dim,
    activation,
    prog_args.gc_per_block,
    prog_args.dropout,
    prog_args.num_pool,
    prog_args.linkpred,
    prog_args.batch_size,
    "meanpool",
    assign_dim,
    prog_args.pool_ratio,
    []
)

print("model init finished")
print("MODEL:::::::", prog_args.method)
if prog_args.cuda:
    model = model.cuda()

=============================================prog_args==============================================
Namespace(dataset='GTmining_6_6', pool_ratio=0.1, num_pool=1, cuda=1, lr=0.0001, clip=1.0, batch_size=1, epoch=500, train_ratio=0.7, test_ratio=0.1, n_worker=2, gc_per_block=3, dropout=0.2, method='diffpool', bn=True, bias=True, save_dir='./model_param', load_epoch=-1, data_mode='default', linkpred=False, hidden_dim=64, embedding_dim=64)
================================================加载数据================================================
++++++++++ STATISTICS ABOUT THE DATASET ++++++++++
dataset feature dimension is 7
dataset label dimension is 10
the max num node is 655
number of graphs is 19896
++++++++++MODEL STATISTICS++++++++
model hidden dim is 64
model embedding dim for graph instance embedding 64
initial batched pool graph dim is 65
model init finished
MODEL::::::: diffpool


In [7]:
# train_dataloader  validation_dataloader  test_dataloader  nova_dataloader
# train  validation  test  nova

id_card_protein = {}
with open(f'../data/dl_data/GTB/fold1/train/Predict_correspond_information.txt', 'r')as f:
    for dd in f.readlines():
        dd = dd.split('\n')[0].split('===')
        id_card_protein[dd[1]] = dd[0]

logger = train(
    train_dataloader, model, prog_args, id_card_protein
)

print("Ending!!!")


EPOCH ###### 0 ######


100%|██████████| 12507/12507 [07:04<00:00, 29.45it/s]

Ending!!!
